<a href="https://colab.research.google.com/github/swarubm/OWN-PROJECTS/blob/main/fintech%20ml%20project%20without%20using%20real%20dataset%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import numpy as np # Import the numpy library for numerical operations
import pandas as pd # Import the pandas library for data manipulation and analysis
from sklearn.model_selection import train_test_split # Import 'train_test_split' for dividing data into training and testing sets
from sklearn.linear_model import LogisticRegression # Import the Logistic Regression model for classification
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report # Import metrics to evaluate model performance

In [10]:
np.random.seed(42) # Set a random seed to ensure reproducibility of the generated data

n_samples = 10000 # Define the total number of samples for the synthetic dataset

# Generate synthetic features for transactions
transaction_amount = np.random.exponential(scale=2000, size=n_samples) # Generate transaction amounts following an exponential distribution
transaction_time = np.random.randint(0, 24, n_samples) # Generate transaction times (hours) randomly between 0 and 23
location_mismatch = np.random.choice([0, 1], size=n_samples, p=[0.95, 0.05]) # Generate location mismatch (0 for no, 1 for yes) with a 5% chance of mismatch

# Define the target variable (fraud = 1, normal = 0) based on multiple conditions
# Fraud is more likely if amount is high, time is late, or location mismatch exists
fraud = (
    (transaction_amount > 5000).astype(int) + # Assign 1 if amount is greater than 5000 (a strong signal for fraud)
    (transaction_time > 22).astype(int) + # Assign 1 if transaction time is after 10 PM (another strong signal)
    (location_mismatch == 1).astype(int) # Assign 1 if a location mismatch occurs (a third strong signal)
)

# A transaction is considered fraudulent if at least two strong signals are present
fraud = (fraud >= 2).astype(int) # Convert the sum of signals into a binary fraud indicator (1 if sum >= 2, else 0)

# Create a pandas DataFrame from the generated data
df = pd.DataFrame({ # Initialize a DataFrame
    "amount": transaction_amount, # Add transaction amount as a column
    "time": transaction_time, # Add transaction time as a column
    "location_mismatch": location_mismatch, # Add location mismatch as a column
    "fraud": fraud # Add the determined fraud status as the target column
})

# Display the count of fraudulent vs. non-fraudulent transactions
df["fraud"].value_counts() # Show the distribution of fraud (1) and non-fraud (0) instances

,count
fraud,
0,9897
1,103


In [11]:
df.head() # Display the first 5 rows of the DataFrame to inspect the generated data structure and content

,amount,time,location_mismatch,fraud
0,938.536180,22,0,0
1,6020.242862,3,0,0
2,2633.491387,17,0,0
3,1825.885108,4,1,0
4,339.249741,15,0,0


In [12]:
X = df.drop("fraud", axis=1) # Create the feature set (X) by removing the 'fraud' column from the DataFrame
y = df["fraud"] # Create the target variable (y) containing only the 'fraud' column

# Split the dataset into training and testing sets
# 80% of the data will be used for training, and 20% for testing
# 'random_state=42' ensures that the split is consistent across runs for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42 # Perform the split
)

In [13]:
model = LogisticRegression() # Initialize an instance of the Logistic Regression model
model.fit(X_train, y_train) # Train the model using the training features (X_train) and corresponding target values (y_train)

y_pred = model.predict(X_test) # Use the trained model to make predictions on the unseen test features (X_test)

In [14]:
print(y_pred) # Print the array of predicted labels (0 or 1) for the test set

[0 0 0 ... 0 0 0]


In [15]:
print("Accuracy:", accuracy_score(y_test, y_pred)) # Calculate and print the overall accuracy of the model's predictions
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred)) # Calculate and print the confusion matrix, showing true positives/negatives and false positives/negatives
print("\nClassification Report:\n", classification_report(y_test, y_pred)) # Generate and print a detailed classification report including precision, recall, and f1-score

Accuracy: 0.9905

Confusion Matrix:
 [[1973    3]
 [  16    8]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00      1976
           1       0.73      0.33      0.46        24

    accuracy                           0.99      2000
   macro avg       0.86      0.67      0.73      2000
weighted avg       0.99      0.99      0.99      2000



In [16]:
# Create a sample transaction for prediction
sample = pd.DataFrame({ # Initialize a new DataFrame for a single sample transaction
    "amount": [8000], # Set the transaction amount for the sample
    "time": [23], # Set the transaction time (hour) for the sample
    "location_mismatch": [1] # Set the location mismatch status for the sample
})

prediction = model.predict(sample) # Predict the fraud status (0 or 1) for the sample transaction using the trained model
probability = model.predict_proba(sample) # Get the predicted probabilities for each class (non-fraud and fraud) for the sample

print("Prediction (1 = Fraud):", prediction[0]) # Print the predicted class (0 for not fraud, 1 for fraud)
print("Fraud Probability:", probability[0][1]) # Print the probability of the transaction being fraudulent (the probability of class 1)

Prediction (1 = Fraud): 1
Fraud Probability: 0.9664894318581132
